# Space X Falcon 9 First Stage Landing Prediction
## Lab 2: Web scraping Falcon 9 launch records with BeautifulSoup

The launch records are scraped from the Wikipedia page
*List of Falcon 9 and Falcon Heavy launches*.

In [1]:
import sys, re, unicodedata
import requests
import pandas as pd
from bs4 import BeautifulSoup

### Helper functions used to clean each scraped cell

In [2]:
def date_time(table_cells):
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    return ''.join([booster_version for i, booster_version
                    in enumerate(table_cells.strings) if i % 2 == 0][0:-1])

def landing_status(table_cells):
    return [i for i in table_cells.strings][0]

def get_mass(table_cells):
    mass = unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass = mass[0:mass.find("kg") + 2]
    else:
        new_mass = 0
    return new_mass

def extract_column_from_header(row):
    if row.br:
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
    colunm_name = ' '.join(row.contents)
    if not colunm_name.strip().isdigit():
        colunm_name = colunm_name.strip()
        return colunm_name

### TASK 1: Request the Falcon 9 launch wiki page from its URL

In [3]:
static_url = ("https://en.wikipedia.org/w/index.php?title="
              "List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922")

# Wikipedia rejects the default python-requests user agent, so a browser one is set.
headers = {'User-Agent': ('Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 '
                          '(KHTML, like Gecko) Chrome/120.0 Safari/537.36')}
response = requests.get(static_url, headers=headers)
print("status code:", response.status_code)

status code: 200


In [4]:
soup = BeautifulSoup(response.text, 'html.parser')
soup.title

<title>List of Falcon 9 and Falcon Heavy launches - Wikipedia</title>

### TASK 2: Extract all column names from the HTML table header

In [5]:
html_tables = soup.find_all('table')
print("tables found:", len(html_tables))
first_launch_table = html_tables[2]

tables found: 25


In [6]:
column_names = []
for th in first_launch_table.find_all('th'):
    name = extract_column_from_header(th)
    if name is not None and len(name) > 0:
        column_names.append(name)
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


### TASK 3: Create a dataframe by parsing the launch HTML tables

In [7]:
launch_dict = dict.fromkeys(column_names)
del launch_dict['Date and time ( )']
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
launch_dict['Version Booster'] = []
launch_dict['Booster landing'] = []
launch_dict['Date'] = []
launch_dict['Time'] = []
list(launch_dict.keys())

['Flight No.',
 'Launch site',
 'Payload',
 'Payload mass',
 'Orbit',
 'Customer',
 'Launch outcome',
 'Version Booster',
 'Booster landing',
 'Date',
 'Time']

In [8]:
extracted_row = 0
for table_number, table in enumerate(
        soup.find_all('table', "wikitable plainrowheaders collapsible")):
    for rows in table.find_all("tr"):
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip()
            flag = flight_number.isdigit()
        else:
            flag = False
        row = rows.find_all('td')
        if flag:
            extracted_row += 1
            launch_dict['Flight No.'].append(flight_number)

            datatimelist = date_time(row[0])
            launch_dict['Date'].append(datatimelist[0].strip(','))
            launch_dict['Time'].append(datatimelist[1])

            bv = booster_version(row[1])
            if not bv:
                bv = row[1].a.string
            launch_dict['Version Booster'].append(bv)

            launch_dict['Launch site'].append(row[2].a.string if row[2].a else row[2].text.strip())
            launch_dict['Payload'].append(row[3].a.string if row[3].a else row[3].text.strip())
            launch_dict['Payload mass'].append(get_mass(row[4]))
            launch_dict['Orbit'].append(row[5].a.string if row[5].a else row[5].text.strip())
            launch_dict['Customer'].append(row[6].a.string if row[6].a else row[6].text.strip())
            launch_dict['Launch outcome'].append(list(row[7].strings)[0])
            launch_dict['Booster landing'].append(landing_status(row[8]))

print("rows extracted:", extracted_row)

rows extracted: 121


In [9]:
df = pd.DataFrame({key: pd.Series(value) for key, value in launch_dict.items()})
print(df.shape)
df.head()

(121, 11)


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,F9 v1.07B0003.1,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0,LEO,NASA,Success,F9 v1.07B0004.1,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525 kg,LEO,NASA,Success,F9 v1.07B0005.1,No,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,"4,700 kg",LEO,NASA,Success,F9 v1.07B0006.1,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,"4,877 kg",LEO,NASA,Success,F9 v1.07B0007.1,No,1 March 2013,15:10


In [10]:
df.to_csv('spacex_web_scraped.csv', index=False)
print("saved spacex_web_scraped.csv")

saved spacex_web_scraped.csv
